In [ ]:
from core import altopt, plot_altopt
from core import DataContext, ParameterContext
import os
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif', 'font.size': 12})

In [ ]:
DATA_DIR = r"DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_8"

i = 3
for j in range(0,4,2):
    curr_time = 41.0 + j
    DATASET3 = DataContext(
        sim_time = os.path.join(DATA_DIR, "t_array.csv"),
        sim_freq = os.path.join(DATA_DIR, "delz_MHz.csv"),
        sim_by = os.path.join(DATA_DIR, "dely_MHz.csv"),
        sim_intensities = sorted([
            os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.startswith("transH_t_vs_Bz_alpha2500_delx_0_dely_") and f.endswith(".csv")
        ], key=lambda x: float(x.split("_dely_")[1].split(".csv")[0])),
        exp_time = r"DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\t_array_ Copy.csv",
        exp_freq_axis = r"DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\Bz_Y_MHz.csv",
        exp_data = f"DataFiles_to_Dinesh_Pranav\\Data_files\\Experiment\\26_03_2026_CW_data\\Bz_V_Readout_By_{i}.csv",
        save_path = f"Results\\Dataset_8\\29_Mar\\Test_{i+1}_Dataset_8_By_{i}_from_{curr_time}_microseconds_0.2_step_Adaptive_Bias_Selection",
        interpolator=r"DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_8\gpu_sim_interpolator_new_normalisation_linear_0.3_threshold.npz",
        Aligned = False,
        sim_pulse_thresh=0.3,
        exp_pulse_thresh=0.3
    )
    DEFAULT_ALTOPT_PARAMS = ParameterContext(
        num_iter=10,
        curr_time = curr_time,
        max_time = 70.0,
        tol_bz=1e-20,
        tol_by=1e-20,
        t_step = 0.2,
        fixed_by_estimate=0.0,
        B_unk_bound_transverse_upper=0.4,
        fixed_bz_estimate=0.0,
        print_plot = False,
        Est_First_Z= False
        )
    sign = "plus" if DEFAULT_ALTOPT_PARAMS.fixed_bz_estimate >= 0 else "minus"
    Test = r"Dataset_7_Starting with" + str(DEFAULT_ALTOPT_PARAMS.curr_time) + r" us_T Step " + str(DEFAULT_ALTOPT_PARAMS.t_step) + r" us_" + sign + r"_" + str(np.abs(DEFAULT_ALTOPT_PARAMS.fixed_bz_estimate)) + r"_initial estimate"
    save_path = sign + r"_" + str(np.abs(DEFAULT_ALTOPT_PARAMS.fixed_bz_estimate)) + r"_initial_estimate"
    DEFAULT_ALTOPT_PARAMS.Test = Test

    results = altopt(DATASET3, DEFAULT_ALTOPT_PARAMS)
    plot_altopt(DATASET3, *results)

New Plotting 

1. I need to create a concatenated array containing all the means and std devs, and also need to give it a savepath, which is fine

In [ ]:
curr_time = 50.0
directory = f"Results\\Dataset_7\\25_Mar\\Start_Time_Test\\Test_Dataset_7_from_{curr_time}_microseconds_0.2_step_Adaptive_Bias_Selection"
result_longitudinal_list = sorted([
        os.path.join(directory, f) for f in os.listdir(directory) if f.startswith(f"longitudinal_estimation_results_Dataset_7_Starting with{curr_time} us_T Step 0.2 us_plus_0.0_initial estimate_26_03_2026__") and f.endswith(r".npz")
    ], key=lambda x: float(x.split(r"26_03_2026__")[1].split(r".npz")[0]))
result_transverse_list = sorted([
        os.path.join(directory, f) for f in os.listdir(directory) if f.startswith(f"transverse_field_estimation_results_Dataset_7_Starting with{curr_time} us_T Step 0.2 us_plus_0.0_initial estimate_26_03_2026__") and f.endswith(r".npz")
    ], key=lambda x: float(x.split(r"26_03_2026__")[1].split(r".npz")[0]))

longitudinal_estimation_results = [np.load(f, allow_pickle=True) for f in result_longitudinal_list]
transverse_field_estimation_results = [np.load(f, allow_pickle=True) for f in result_transverse_list]
print(len(longitudinal_estimation_results), len(transverse_field_estimation_results))

bz_full_traj = []
bz_final_estimate = []
by_full_traj = []
by_final_estimate = []
bz_full_upper_bound = []
bz_full_lower_bound = []
by_full_upper_bound = []
by_full_lower_bound = []
iters_per_block = 0
save_path = directory
for i in range(len(longitudinal_estimation_results)):
    if i==0:
        iters_per_block = longitudinal_estimation_results[i]["est"].shape[0]
        print(f"Iters per block: {iters_per_block}")
    bz_full_traj += list(longitudinal_estimation_results[i]["est"])
    bz_final_estimate.append(longitudinal_estimation_results[i]["est"][-1])
    by_full_traj += list(transverse_field_estimation_results[i]["est"])

    by_final_estimate.append(transverse_field_estimation_results[i]["est"][-1])

    #Since, sigma is not a good measure for confidence interval, let us use interval estimation of 90% probability for each posterior, the posterior is a numpy array 
    bz_posterior = list(longitudinal_estimation_results[i]["posteriors"])
    by_posterior = list(transverse_field_estimation_results[i]["posteriors"])
    bz_grid = list(longitudinal_estimation_results[i]["bgrids"])
    by_grid = list(transverse_field_estimation_results[i]["bgrids"])\

    #since first element is the prior, we gotta remove that
    for posterior, grid in zip(bz_posterior[1:], bz_grid[1:]):
        dB: np.ndarray = grid[1:] - grid[:-1]
        pdf_mass = np.zeros_like(posterior)
        pdf_mass[:-1] = posterior[:-1] * dB
        pdf_mass[-1] = posterior[-1] * dB[-1]
        cdf = np.cumsum(pdf_mass)
        cdf = cdf/cdf[-1]
        bz_full_upper_bound += [grid[np.argmax(cdf >= 0.99)]]
        bz_full_lower_bound += [grid[np.argmax(cdf >= 0.01)]]
    for posterior, grid in zip(by_posterior[1:], by_grid[1:]):
        dB: np.ndarray = grid[1:] - grid[:-1]
        pdf_mass = np.zeros_like(posterior)
        pdf_mass[:-1] = posterior[:-1] * dB
        pdf_mass[-1] = posterior[-1] * dB[-1]
        cdf = np.cumsum(pdf_mass)
        cdf = cdf/cdf[-1]
        by_full_upper_bound += [grid[np.argmax(cdf >= 0.99)]]
        by_full_lower_bound += [grid[np.argmax(cdf >= 0.01)]]

    if i == 9:
        break

print(len(bz_full_traj), len(by_full_traj), len(bz_full_upper_bound), len(bz_full_lower_bound), len(by_full_upper_bound), len(by_full_lower_bound))
bz_full_traj, by_full_traj = np.array(bz_full_traj), np.array(by_full_traj)
    

In [ ]:
def plot_intra_iteration_convergence(bz_full_traj, 
                                    bz_full_upper_bound,
                                    bz_full_lower_bound,    
                                    bz_final_estimate, 
                                    by_full_traj,
                                    by_full_upper_bound,
                                    by_full_lower_bound,
                                    by_final_estimate, iters_per_block, save_path=None):
    """
    Plots the continuous, intra-iteration evolution of the Bayesian estimates.
    
    Parameters:
    - bz_full_traj, bz_full_std: 1D numpy arrays of length (N_total_steps)
    - by_full_traj, by_full_std: 1D numpy arrays of length (N_total_steps)
    - iters_per_block: Integer (e.g., if you run 100 KL steps per alternating iteration)
    """
    bz_full_traj, bz_full_upper_bound, bz_full_lower_bound, by_full_traj, by_full_upper_bound, by_full_lower_bound, bz_final_estimate, by_final_estimate = np.array(bz_full_traj)/0.7, np.array(bz_full_upper_bound)/0.7, np.array(bz_full_lower_bound)/0.7, np.array(by_full_traj)/0.7, np.array(by_full_upper_bound)/0.7, np.array(by_full_lower_bound)/0.7, np.array(bz_final_estimate)/0.7, np.array(by_final_estimate)/0.7
    # Create an artificial x-axis where 1 "unit" is a full Alternating Optimization block
    n_total_steps = len(bz_full_traj)
    x_axis = np.arange(n_total_steps) / iters_per_block
    num_blocks = int(np.ceil(n_total_steps / iters_per_block))
    
    plt.rcParams.update({'font.family': 'serif', 'font.size': 10})
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={'hspace': 0.1})

    # Plot Bz
    ax1.plot(x_axis, bz_full_traj, color='#1f77b4', linewidth=1.5, label='Intra-iteration MAP')
    ax1.plot(range(1,num_blocks+1), bz_final_estimate, '--', color= "#487494",marker = 'o', markersize=4, label='Final Estimate')
    ax1.fill_between(x_axis, bz_full_lower_bound, bz_full_upper_bound, color='#1f77b4', alpha=0.3, label=r'98% Credible Interval')
    # ax1.fill_between(x_axis, bz_full_traj - 3*bz_full_std, bz_full_traj + 3*bz_full_std, color='#1f77b4', alpha=0.1, label=r'$\pm 3\sigma$')
    ax1.set_ylabel(r'$B_z$ Estimate (Gauss)')
    ax1.grid(True, linestyle=':', alpha=0.5)
    ax1.set_ylim(-1/0.7, 1/0.7)
    # ax1.autoscale(enable=True, axis='y', tight=True)
    ax1.legend(loc='upper right', framealpha=0.9, ncol=2)

    # Plot By
    ax2.plot(x_axis, by_full_traj, color='#d62728', linewidth=1.5, label='Intra-iteration MAP')
    ax2.plot(range(1,num_blocks+1), by_final_estimate, '--', color= "#487494",marker = 'o', markersize=4, label='Final Estimate')
    ax2.fill_between(x_axis, by_full_lower_bound, by_full_upper_bound, color='#d62728', alpha=0.3, label=r'98% Credible Interval')
    # ax2.fill_between(x_axis, by_full_traj - 3*by_full_std, by_full_traj + 3*by_full_std, color='#d62728', alpha=0.1, label=r'$\pm 3\sigma$')
    ax2.set_ylabel(r'$B_y$ Estimate (Gauss)')
    ax2.set_xlabel('Alternating Optimization Iteration')
    ax2.grid(True, linestyle=':', alpha=0.5)
    ax2.set_ylim(0, 0.4/0.7)
    # ax2.autoscale(enable=True, axis='y', tight=True)
    ax2.legend(loc='upper right', framealpha=0.9, ncol=2)

    # Formatting visual dividers
    num_blocks = int(np.ceil(n_total_steps / iters_per_block))
    for i in range(num_blocks + 1):
        ax1.axvline(i, color='black', linestyle='--', alpha=0.7)
        ax2.axvline(i, color='black', linestyle='--', alpha = 0.7)
        
        '''# Optional: Shade the background to show which variable is actively updating
        if i < num_blocks:
            if i % 2 == 0:
                ax1.axvspan(i, i+1, color='#1f77b4', alpha=0.05) # Bz Active
            else:
                ax2.axvspan(i, i+1, color='#d62728', alpha=0.05) # By Active'''

    ax2.set_xlim(0, num_blocks)
    ax2.set_xticks(np.arange(num_blocks + 1))
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_intra_iteration_convergence(bz_full_traj, bz_full_upper_bound, bz_full_lower_bound, bz_final_estimate, by_full_traj, by_full_upper_bound, by_full_lower_bound, by_final_estimate, iters_per_block, save_path=os.path.join(save_path, "intra_iteration_convergence.png"))